In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmirajravi/rt-detr-model-for-training/best (1).pt
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/README.dataset.txt
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/README.roboflow.txt
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/029_jpg.rf.b811adb8d70cbf53b813101cf61504a8.jpg
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/124_jpg.rf.d9c4d06b882eae670ef3f460a5c55780.jpg
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/0940_jpeg.rf.ef7215c5407dcfed4a238646772c000e.jpg
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/006_jpg.rf.af71de019071381de68bc6bb8da9b522.jpg
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/096_jpg.rf.27ea7c5bcf6165ea15f1b752de5249cc.jpg
/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid/088_jpg.rf.69ec523beb3dbdefca25e48e127

In [2]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.3 MB/s eta 0:00:00


In [3]:
import yaml

# Define the exact Kaggle paths and your 24 classes
data = {
    'train': '/kaggle/input/multiclass-waste-dataset-rt-detr/train/images',
    'val': '/kaggle/input/multiclass-waste-dataset-rt-detr/valid/images',
    'test': '/kaggle/input/multiclass-waste-dataset-rt-detr/test/images',
    'nc': 24,
    'names': [
        'cardboard', 'carton packaging', 'cigarette', 'clean paper', 
        'clear plastic', 'contaminated paper', 'food packaging', 'food scraps', 
        'glass', 'medical waste', 'metal', 'paper bag', 'paper cup', 
        'plastic bottle', 'plastic container', 'plastic cup', 'plastic lid', 
        'plastic packaging', 'plastic utensil', 'printed cardboard', 
        'sanitary waste', 'straw', 'styrofoam', 'wood'
    ]
}

# Save it to the working directory
yaml_path = '/kaggle/working/waste_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
    
print(f"Configuration saved successfully to {yaml_path}")

Configuration saved successfully to /kaggle/working/waste_data.yaml


In [4]:
import os

print("--- Searching for your .pt model file ---")
model_found = False

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.pt'):
            print(f"SUCCESS! Your exact model path is:\n'{os.path.join(root, file)}'")
            model_found = True

if not model_found:
    print("ERROR: No .pt files found. Check if your model dataset is successfully attached to the notebook side panel.")

--- Searching for your .pt model file ---
SUCCESS! Your exact model path is:
'/kaggle/input/datasets/lakshmirajravi/rt-detr-model-for-training/best (1).pt'


In [5]:
import os

print("--- Scanning Kaggle Input for Dataset Structure ---")
for root, dirs, files in os.walk('/kaggle/input'):
    # We look for folders containing 'train', 'val', or 'valid'
    if any(x in root.lower() for x in ['train', 'valid', 'val']) and not dirs:
        # Print directory if it contains files (like images)
        if len(files) > 0:
            print(f"Found data folder: {root} (Contains {len(files)} files)")

--- Scanning Kaggle Input for Dataset Structure ---
Found data folder: /kaggle/input/datasets/lakshmirajravi/rt-detr-model-for-training (Contains 1 files)
Found data folder: /kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid (Contains 19 files)
Found data folder: /kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/train (Contains 2200 files)


In [6]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict
import yaml
from ultralytics import RTDETR

def convert_coco_to_yolo(input_dir, split_name):
    print(f"⏳ Converting {split_name} annotations from COCO to YOLO format...")
    
    # 1. Find the JSON file in the directory
    json_path = None
    for f in os.listdir(input_dir):
        if f.endswith('.json'):
            json_path = os.path.join(input_dir, f)
            break
            
    if not json_path:
        print(f"⚠️ No JSON found in {input_dir}.")
        return
        
    with open(json_path) as f:
        coco = json.load(f)
        
    # 2. Filter out the supercategory (ID 0) and map the 24 valid classes to 0-indexed YOLO IDs
    valid_cats = sorted([cat for cat in coco['categories'] if cat['id'] != 0], key=lambda x: x['name'].lower())
    cat_mapping = {cat['id']: i for i, cat in enumerate(valid_cats)}
    
    # 3. Setup the parallel folder structure YOLO expects
    out_img_dir = Path(f'/kaggle/working/waste_dataset/images/{split_name}')
    out_lbl_dir = Path(f'/kaggle/working/waste_dataset/labels/{split_name}')
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    # 4. Group annotations by image
    anns_by_img = defaultdict(list)
    for ann in coco['annotations']:
        anns_by_img[ann['image_id']].append(ann)
        
    # 5. Copy images and generate .txt label files
    for img in coco['images']:
        img_id = img['id']
        file_name = Path(img['file_name']).name
        src_img = os.path.join(input_dir, file_name)
        
        if os.path.exists(src_img):
            # Safely copy image to the new YOLO-structured folder
            dst_img = out_img_dir / file_name
            if not dst_img.exists():
                shutil.copy2(src_img, dst_img)
                
            # Create the corresponding YOLO .txt file
            txt_path = out_lbl_dir / f"{Path(file_name).stem}.txt"
            with open(txt_path, 'w') as txt_file:
                for ann in anns_by_img.get(img_id, []):
                    cat_id = ann['category_id']
                    if cat_id not in cat_mapping:
                        continue # Skip supercategory
                        
                    cls_id = cat_mapping[cat_id]
                    dw = 1.0 / img['width']
                    dh = 1.0 / img['height']
                    
                    # Convert COCO bbox to normalized YOLO bbox
                    x_center = (ann['bbox'][0] + ann['bbox'][2] / 2.0) * dw
                    y_center = (ann['bbox'][1] + ann['bbox'][3] / 2.0) * dh
                    w = ann['bbox'][2] * dw
                    h = ann['bbox'][3] * dh
                    
                    txt_file.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
                    
    print(f"✅ {split_name} conversion complete!")

# --- EXECUTION STEPS ---

# Step 1: Convert both training and validation sets
convert_coco_to_yolo('/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/train', 'train')
convert_coco_to_yolo('/kaggle/input/datasets/lakshmirajravi/multiclass-waste-dataset-rt-detr/valid', 'val')

# Step 2: Generate the corrected data.yaml pointing to the new converted dataset
data = {
    'train': '/kaggle/working/waste_dataset/images/train', 
    'val': '/kaggle/working/waste_dataset/images/val',   
    'nc': 24,
    'names': [
        'cardboard', 'carton packaging', 'cigarette', 'clean paper', 
        'clear plastic', 'contaminated paper', 'food packaging', 'food scraps', 
        'glass', 'medical waste', 'metal', 'paper bag', 'paper cup', 
        'plastic bottle', 'plastic container', 'plastic cup', 'plastic lid', 
        'plastic packaging', 'plastic utensil', 'printed cardboard', 
        'sanitary waste', 'straw', 'styrofoam', 'wood'
    ]
}

yaml_path = '/kaggle/working/waste_data_converted.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
    
print("✅ Converted waste_data_converted.yaml created successfully!")

# Step 3: Load Model and Train for 150 Epochs
model_path = '/kaggle/input/datasets/lakshmirajravi/rt-detr-model-for-training/best (1).pt'
model = RTDETR(model_path)

print("🚀 Starting training on the converted dataset for 150 epochs...")
results = model.train(
    data=yaml_path, 
    epochs=150,                    
    imgsz=640,                     
    batch=16,                      
    device=0,                      
    project='/kaggle/working/waste_detection', 
    name='rtdetr_multiclass',      
    save=True,             
    save_period=5                  
)
print("🎉 Training complete!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
⏳ Converting train annotations from COCO to YOLO format...
✅ train conversion complete!
⏳ Converting val annotations from COCO to YOLO format...
✅ val conversion complete!
✅ Converted waste_data_converted.yaml created successfully!
🚀 Starting training on the converted dataset for 150 epochs...
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      11.8G     0.7188      2.957     0.5877         61        640: 100% ━━━━━━━━━━━━ 138/138 1.2s/it 2:47
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.1s/it 3.1s
                   all         18        113      0.722      0.179      0.178      0.129

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/150      12.4G     0.3664      1.068     0.2525         44        640: 100% ━━━━━━━━━━━━ 138/138 1.2s/it 2:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         18        113      0.599       0.42      0.446      0.343

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

    148/150      12.7G      0.068     0.1356    0.04097         30        640: 100% ━━━━━━━━━━━━ 138/138 1.2s/it 2:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         18        113      0.943      0.683      0.781      0.645

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    149/150      12.8G    0.06753     0.1355    0.04132         33        640: 100% ━━━━━━━━━━━━ 138/138 1.2s/it 2:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         18        113      0.943      0.677      0.785      0.646

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    150/150      12.7G    0.06741     0.1367    0.04083         23        640: 100% ━━━━━━━━━━━━ 138/138 1.2s/it 2:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         18        113      0.942      0.684      0.782      0.646

150 epochs completed in 6.758 hours.
Optimizer stripped from /kaggle/working/waste_detection/rtdetr_multiclass/weights/last.pt, 66.3MB
Optimizer stripped from /kaggle/working/waste_detection/rtdetr_multiclass/weights/best.pt, 66.3MB

Validating /kaggle/working/waste_detection/rtdetr_multiclass/weights/best.pt...
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 310 layers, 32,033,060 parameters, 0 gradients, 103.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.1it/s 0.5s
                   all         18        113      0.828      0.73